In [ ]:
import os

In [ ]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

In [ ]:
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

In [ ]:
from flow import execute

In [ ]:
chunks = execute()

In [ ]:
chunks

In [ ]:
print(len(chunks))

In [ ]:
import re
from sentence_transformers import SentenceTransformer, util

def clean_text(text):
    return text.strip()

chunks = [clean_text(chunk) for chunk in chunks]
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks, convert_to_tensor=True)
reference_clauses = [
    "The Contractor may offer additional maintenance services if eligible.",
    "Future support services may enhance bid eligibility.",
    "Optional maintenance support may be provided to enhance bid competitiveness."
]
ref_embeddings = model.encode(reference_clauses, convert_to_tensor=True)
threshold = 0.4
matched_chunks = []

for chunk, embedding in zip(chunks, chunk_embeddings):
    cosine_scores = util.cos_sim(embedding, ref_embeddings)
    max_score = cosine_scores.max().item()
    if max_score >= threshold:
        matched_chunks.append((chunk, max_score))
print("Matched Chunks:")
for chunk, score in matched_chunks:
    print(f"Chunk: \"{chunk}\" | Similarity Score: {score:.2f}")


In [ ]:
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt_template = """
You are a proposal expert. Here is a contract clause that was identified as beneficial:

"{clause}"

Summarize the clause in simple language and explain what actions I can take to improve my proposal based on this clause.
Provide clear, short, and actionable advice.
"""

prompt = PromptTemplate(
    input_variables=["clause"],
    template=prompt_template
)

chain = prompt | llm | StrOutputParser()

print("Matched Chunks and Improvement Advice:")
for clause, score in matched_chunks:
    response = chain.invoke({"clause": clause})
    print(f"Chunk: \"{clause}\" | Similarity Score: {score:.2f}")
    print("Improvement Advice:")
    print(response)
    print("-" * 80)
